In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:


import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# ============================================================
# 1. LOAD
# ============================================================
qs = pd.read_csv('/content/drive/MyDrive/qs-world-rankings-2025.csv', encoding='utf-8-sig')
the = pd.read_csv('/content/drive/MyDrive/THE World University Rankings 2016-2026.csv')

qs = qs.drop_duplicates()
the = the.drop_duplicates()

# ============================================================
# ============================================================
# 2. STANDARDIZE TEXT FIELDS
# ============================================================
qs['Institution Name'] = qs['Institution Name'].str.strip().str.lower()
qs['Institution Name'] = qs['Institution Name'].str.replace(r'\s*\([^)]*\)', '', regex=True).str.strip()  # <-- NEW LINE
qs['Location Full']    = qs['Location Full'].str.strip().str.lower()
the['Name']    = the['Name'].str.strip().str.lower()
the['Country'] = the['Country'].str.strip().str.lower()

# ============================================================
# 3. FILTER THE TO ONE YEAR (2025) SO IT LINES UP WITH QS
# ============================================================
the_year = the[the['Year'] == 2025].copy()

# ============================================================
# 4. CLEAN NUMERIC FIELDS BEFORE MERGE/SCALING
# ============================================================
# THE's "International Students" is a percentage string like "26%"
the_year['International Students'] = (
    the_year['International Students'].astype(str).str.replace('%', '', regex=False)
)
the_year['International Students'] = pd.to_numeric(the_year['International Students'], errors='coerce')

# Avoid _x/_y suffix confusion since both files have "International Students"
qs = qs.rename(columns={'International Students': 'QS_International_Students_Pct'})
the_year = the_year.rename(columns={'International Students': 'THE_International_Students_Pct'})

# QS Overall Score and Research Quality sometimes use '-' for missing values
qs['QS Overall Score'] = pd.to_numeric(qs['QS Overall Score'], errors='coerce')
the_year['Research Quality'] = pd.to_numeric(the_year['Research Quality'], errors='coerce')

def clean_population_range(s):
    if pd.isna(s):
        return None
    s = str(s).replace(',', '').strip().lower()
    if s == 'n/a' or s == '-':
        return None
    if '-' in s:
        parts = s.split('-')
        try:
            return (float(parts[0]) + float(parts[1])) / 2
        except ValueError:
            return None
    elif '+' in s:
        try:
            return float(s.replace('+', ''))
        except ValueError:
            return None
    elif '~' in s:
        try:
            return float(s.replace('~', ''))
        except ValueError:
            return None
    try:
        return float(s)
    except ValueError:
        return None

the_year['Student_Population_Cleaned'] = the_year['Student Population'].apply(clean_population_range)

# ============================================================
# 5. MERGE (inner join = only rows present in BOTH datasets)
# ============================================================
merged = pd.merge(
    qs, the_year,
    left_on=['Institution Name', 'Location Full'],
    right_on=['Name', 'Country'],
    how='inner'
)

print("Rows after merge:", merged.shape[0])

if merged.shape[0] == 0:
    raise ValueError("Merge produced 0 rows — check that institution/country fields overlap after cleaning.")

# ============================================================
# 6. BUILD THE FEATURES NEEDED FOR PERFORMANCE INDEX
# ============================================================
merged['Global_Rank_Score']           = merged['QS Overall Score']
merged['Research_Productivity_Index'] = merged['Research Quality']
merged['Intl_Student_Percentage']     = merged['THE_International_Students_Pct']

# ---- Force these to numeric, turning any stray '-' or text into NaN ----
for col in ['Global_Rank_Score', 'Research_Productivity_Index', 'Intl_Student_Percentage']:
    merged[col] = pd.to_numeric(merged[col], errors='coerce')

# drop rows missing any of the three inputs the index depends on
merged = merged.dropna(subset=['Global_Rank_Score', 'Research_Productivity_Index', 'Intl_Student_Percentage'])
print("Rows after removing missing/non-numeric scores:", merged.shape[0])

# ============================================================
# 7. SCALE + PERFORMANCE INDEX
# ============================================================
scaler = MinMaxScaler()
merged[['Global_Rank_Score','Research_Productivity_Index','Intl_Student_Percentage']] = scaler.fit_transform(
    merged[['Global_Rank_Score','Research_Productivity_Index','Intl_Student_Percentage']]
)

merged['Performance_Index'] = (
    0.4 * merged['Global_Rank_Score'] +
    0.3 * merged['Research_Productivity_Index'] +
    0.3 * merged['Intl_Student_Percentage']
)

# ============================================================
# 8. CHART
# ============================================================
import plotly.express as px
fig = px.bar(
    merged.sort_values('Performance_Index', ascending=False).head(10),
    x='Institution Name', y='Performance_Index',
    title='Top 10 Universities by Performance Index'
)
fig.show()

# ============================================================
# 9. VERIFICATION
# ============================================================
check_cols = ['Institution Name','Country','Global_Rank_Score',
              'Research_Productivity_Index','Intl_Student_Percentage','Performance_Index']

print("\n--- Nulls in key columns ---")
print(merged[check_cols].isnull().sum())

print("\n--- Duplicate universities ---")
print("Duplicate rows:", merged.duplicated(subset=['Institution Name','Country']).sum())

print("\n--- Row count sanity check ---")
print("QS rows:", qs.shape[0], "| THE 2025 rows:", the_year.shape[0], "| Merged rows:", merged.shape[0])

print("\n--- Score ranges (should be 0 to 1) ---")
print(merged[['Global_Rank_Score','Research_Productivity_Index','Intl_Student_Percentage','Performance_Index']].describe())

print("\n--- Spot check well-known universities ---")
print(merged[merged['Institution Name'].str.contains('oxford|mit|stanford', case=False, na=False)]
      [['Institution Name','Country','Performance_Index']])

print("\n--- Data types ---")
print(merged.dtypes[['Global_Rank_Score','Research_Productivity_Index','Intl_Student_Percentage']])

# ============================================================
# 10. EXPORT FOR TABLEAU
# ============================================================
merged.to_csv('/content/drive/MyDrive/university_final_dataset.csv', index=False)
print("\nSaved to /content/drive/MyDrive/university_final_dataset.csv")

Rows after merge: 728
Rows after removing missing/non-numeric scores: 349



--- Nulls in key columns ---
Institution Name               0
Country                        0
Global_Rank_Score              0
Research_Productivity_Index    0
Intl_Student_Percentage        0
Performance_Index              0
dtype: int64

--- Duplicate universities ---
Duplicate rows: 0

--- Row count sanity check ---
QS rows: 1503 | THE 2025 rows: 2092 | Merged rows: 349

--- Score ranges (should be 0 to 1) ---
       Global_Rank_Score  Research_Productivity_Index  \
count         349.000000                   349.000000   
mean            0.284286                     0.716385   
std             0.243261                     0.205648   
min             0.000000                     0.000000   
25%             0.090909                     0.597714   
50%             0.218434                     0.762286   
75%             0.417929                     0.873143   
max             1.000000                     1.000000   

       Intl_Student_Percentage  Performance_Index  
count          

In [4]:
print(merged[merged['Institution Name'].str.contains('massachusetts institute', case=False, na=False)]
      [['Institution Name','Country','Performance_Index']])

                        Institution Name        Country  Performance_Index
0  massachusetts institute of technology  united states           0.822222


In [ ]:
# Right after: merged = pd.merge(...)   -- BEFORE building Global_Rank_Score
print("Total rows after merge:", merged.shape[0])
print("QS Overall Score missing:", merged['QS Overall Score'].isna().sum())
print("Research Quality missing:", merged['Research Quality'].isna().sum())
print("THE Intl Students missing:", merged['THE_International_Students_Pct'].isna().sum())merged = pd.merge(
    qs, the_year,
    left_on=['Institution Name', 'Location Full'],
    right_on=['Name', 'Country'],
    how='inner'
)

print("Rows after merge:", merged.shape[0])
print("Columns after merge:", merged.columns.tolist())

if merged.shape[0] == 0:
    raise ValueError("Merge produced 0 rows — check that 'Institution Name'/'Location Full' "
                      "and 'Name'/'Country' actually overlap after cleaning.")

Rows after merge: 685
Columns after merge: ['2025 Rank', '2024 Rank', 'Institution Name', 'Location', 'Location Full', 'Size', 'Academic Reputation', 'Employer Reputation', 'Faculty Student', 'Citations per Faculty', 'International Faculty', 'QS_International_Students_Pct', 'International Research Network', 'Employment Outcomes', 'Sustainability', 'QS Overall Score', 'Rank', 'Name', 'Country', 'Student Population', 'Students to Staff Ratio', 'THE_International_Students_Pct', 'Female to Male Ratio', 'Overall Score', 'Teaching', 'Research Environment', 'Research Quality', 'Industry Impact', 'International Outlook', 'Year', 'Student_Population_Cleaned']


In [ ]:
print("=" * 60)
print("MERGE & DATA QUALITY VERIFICATION")
print("=" * 60)

# 1. No nulls anywhere in the final dataset (not just key columns)
total_nulls = merged.isnull().sum().sum()
print(f"\n1. Total nulls across ALL columns: {total_nulls}")
if total_nulls > 0:
    print("   Columns with nulls:")
    print(merged.isnull().sum()[merged.isnull().sum() > 0])

# 2. Confirm both source datasets actually contributed columns
qs_only_cols = [c for c in qs.columns if c in merged.columns]
the_only_cols = [c for c in the_year.columns if c in merged.columns]
print(f"\n2. Columns from QS present in merged: {len(qs_only_cols)}")
print(f"   Columns from THE present in merged: {len(the_only_cols)}")
print(f"   Total merged columns: {merged.shape[1]}")

# 3. No exact duplicate rows
print(f"\n3. Fully duplicate rows: {merged.duplicated().sum()}")

# 4. Every row has a valid match on both sides (no blank institution/country)
print(f"\n4. Blank Institution Name: {(merged['Institution Name'].str.strip() == '').sum()}")
print(f"   Blank Country: {(merged['Country'].str.strip() == '').sum()}")

# 5. Confirm merge actually combined data, not just stacked it
#    (row count should be less than either source, since inner join)
print(f"\n5. QS rows: {qs.shape[0]} | THE 2025 rows: {the_year.shape[0]} | Merged: {merged.shape[0]}")
print(f"   Merged should be <= smaller source. Is it? {merged.shape[0] <= min(qs.shape[0], the_year.shape[0])}")

# 6. Sample check — print 5 random rows to eyeball manually
print("\n6. Random sample of 5 merged rows:")
print(merged[['Institution Name','Country','Global_Rank_Score',
              'Research_Productivity_Index','Intl_Student_Percentage',
              'Performance_Index']].sample(5, random_state=1))

print("\n" + "=" * 60)
print("If all checks above look clean, the dataset is ready for Tableau.")
print("=" * 60)

MERGE & DATA QUALITY VERIFICATION

1. Total nulls across ALL columns: 26
   Columns with nulls:
International Faculty     2
Female to Male Ratio     24
dtype: int64

2. Columns from QS present in merged: 16
   Columns from THE present in merged: 15
   Total merged columns: 35

3. Fully duplicate rows: 0

4. Blank Institution Name: 0
   Blank Country: 0

5. QS rows: 1503 | THE 2025 rows: 2092 | Merged: 349
   Merged should be <= smaller source. Is it? True

6. Random sample of 5 merged rows:
                         Institution Name         Country  Global_Rank_Score  \
192       north carolina state university   united states           0.180556   
256  czech technical university in prague  czech republic           0.095960   
169                 university of antwerp         belgium           0.226010   
67                      boston university   united states           0.470960   
201                   university of miami   united states           0.167929   

     Research_Productiv

In [ ]:
# right after the merge (685 rows), before building Global_Rank_Score etc.
print("QS Overall Score missing/invalid:", merged['QS Overall Score'].isna().sum())
print("Research Quality missing/invalid:", merged['Research Quality'].isna().sum())
print("THE Intl Students missing/invalid:", merged['THE_International_Students_Pct'].isna().sum())

QS Overall Score missing/invalid: 0
Research Quality missing/invalid: 0
THE Intl Students missing/invalid: 0
